# 04 – TF-IDF-Evidence aufbauen

Dieses Notebook berechnet **TF-IDF-Schlüsselbegriffe** aus den BM25-Ergebnissen
und baut daraus strukturierte *Evidence-Strings* für den LLM-Prompt.

**Drei Varianten:**
- `rel` – TF-IDF aus pseudo-relevanten Dokumenten (Top-3 BM25)
- `non_rel` – TF-IDF aus nicht-relevanten Dokumenten (Rang 100–103)
- `contrastive` – Kombination aus beiden (rel + non_rel gemeinsam)

**Voraussetzung:** `03_retrieval_BM25.ipynb` wurde ausgeführt.

**Ausgabe:** `data/evidence_topic_generation.csv`

## 1. Imports

In [1]:
import ast
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

## 2. Retrieval-Ergebnisse laden

Liest `data/retrieval_bm25_topic.csv`. Die Listen-Spalten (`texts`, `docnos`) wurden
als Strings gespeichert und werden mit `ast.literal_eval` zurück in Python-Listen konvertiert.

In [2]:
# Retrieval-Ergebnisse laden
# Die Listen-Spalten wurden als String gespeichert → literal_eval zum Zurückkonvertieren
retrieval_df = pd.read_csv("data/retrieval_bm25_topic.csv")

retrieval_df["texts"] = retrieval_df["texts"].apply(ast.literal_eval)
retrieval_df["docnos"] = retrieval_df["docnos"].apply(ast.literal_eval)

print("Retrieval-Zeilen:", len(retrieval_df))

Retrieval-Zeilen: 752


## 3. TF-IDF-Hilfsfunktionen

- **`tfidf_top_terms()`**: TF-IDF über alle Dokumente gemeinsam, gemittelte Gewichte, Top-N Terme.
  Uni- und Bigramme werden berücksichtigt (`ngram_range=(1,2)`).
- **`tfidf_to_string()`**: Formatiert die Terme als Score-String für den Prompt.
- **`build_evidence()`**: Kombiniert Query und TF-IDF-Terme zum finalen Evidence-String.

In [3]:
# TF-IDF-Hilfsfunktionen
# tfidf_top_terms: berechnet TF-IDF über alle Dokumente gemeinsam,
# mittelt die Gewichte – so werden Terme hervorgehoben, die für die gesamte Dokumentbasis typisch sind.
def tfidf_top_terms(doc_texts, top_n=5):
    doc_texts = [str(t) for t in doc_texts if t and str(t).strip()]
    if not doc_texts:
        return pd.DataFrame(columns=["term", "tfidf_score"])
    vectorizer = TfidfVectorizer(
        stop_words="english",
        ngram_range=(1, 2),   # Uni- und Bigramme
        max_df=0.9,           # Terme, die in >90% der Docs vorkommen, werden ignoriert
        min_df=1,
        max_features=5000
    )
    X = vectorizer.fit_transform(doc_texts)
    scores = np.asarray(X.mean(axis=0)).ravel()  # Gemittelte TF-IDF-Gewichte
    terms = vectorizer.get_feature_names_out()
    df = pd.DataFrame({"term": terms, "tfidf_score": scores})
    return df.sort_values("tfidf_score", ascending=False).head(top_n)

def tfidf_to_string(tfidf_df):
    """Formatiert Top-Terme als lesbaren Score-String für den LLM-Prompt."""
    return ", ".join(
        f"{r.term} ({r.tfidf_score:.3f})"
        for r in tfidf_df.itertuples(index=False)
    )

def build_evidence(query_text: str, tfidf_df: pd.DataFrame) -> str:
    """Kombiniert Query-Text und TF-IDF-Terme zum strukturierten Evidence-String."""
    terms_with_scores = tfidf_to_string(tfidf_df)
    evidence = (
        "Search query:\n"
        f"{query_text}\n\n"
        "Important TF-IDF terms extracted from retrieved documents:\n"
        f"{terms_with_scores}"
    )
    return evidence

## 4. Evidence-Vektoren berechnen und speichern

Für jede Zeile aus den Retrieval-Ergebnissen wird der Evidence-String berechnet.
Zusätzlich wird die `contrastive`-Variante durch Zusammenführen von `rel` und `non_rel`
erzeugt. Erwartetes Ergebnis: 1.128 Zeilen (376 Sessions × 3 Varianten).

In [4]:
# Evidence-Vektoren aufbauen
# rel + non_rel separat verarbeiten; contrastive entsteht durch Merge beider Varianten

TFIDF_TOP_N = 5

evidence_rows = []

for _, row in retrieval_df.iterrows():
    tfidf_df = tfidf_top_terms(row["texts"], top_n=TFIDF_TOP_N)
    evidence = build_evidence(row["query_text"], tfidf_df)

    evidence_rows.append({
        "session_id": row["session_id"],
        "doc_variant": row["doc_variant"],
        "query_text": row["query_text"],
        "docnos": row["docnos"],
        "tfidf_terms": tfidf_to_string(tfidf_df),
        "evidence": evidence
    })

evidence_df = pd.DataFrame(evidence_rows)

# Contrastive-Variante: rel + non_rel Evidence zusammenführen
rel_df = evidence_df[evidence_df["doc_variant"] == "rel"].copy()
non_rel_df = evidence_df[evidence_df["doc_variant"] == "non_rel"].copy()
merged = rel_df.merge(non_rel_df, on="session_id", suffixes=("_rel", "_non"))

contrastive_rows = []
for _, row in merged.iterrows():
    combined_evidence = (
        "[Relevant Documents]\n" + row["evidence_rel"] +
        "\n\n[Non-Relevant Documents]\n" + row["evidence_non"]
    )
    contrastive_rows.append({
        "session_id": row["session_id"],
        "doc_variant": "contrastive",
        "query_text": row["query_text_rel"],
        "docnos": row["docnos_rel"],
        "tfidf_terms": row["tfidf_terms_rel"],
        "evidence": combined_evidence
    })

contrastive_df = pd.DataFrame(contrastive_rows)
evidence_df = pd.concat([evidence_df, contrastive_df], ignore_index=True)

import os
os.makedirs("data", exist_ok=True)
evidence_df.to_csv("data/evidence_topic_generation.csv", index=False)

print("Anzahl Evidence-Zeilen:", len(evidence_df))
display(evidence_df.head())
print(evidence_df.groupby("doc_variant").size())

Anzahl Evidence-Zeilen: 1128


,session_id,doc_variant,query_text,docnos,tfidf_terms,evidence
0,d19e7b4de358bcfeb89e06095dc26419,rel,peer to peer communication,"[33894675, 301010747, 300892348]","disability (0.130), peer peer (0.123), communi...",Search query:\npeer to peer communication\n\nI...
1,d19e7b4de358bcfeb89e06095dc26419,non_rel,peer to peer communication,"[294182816, 161854289, 310370055, 45476638]","learning (0.142), peer (0.129), students (0.10...",Search query:\npeer to peer communication\n\nI...
2,00dea7cb26b0df14733b1aa2e48d4189,rel,wattpad reading comprehension,"[11012449, 151378454, 122513221]","respondents (0.153), attitude (0.095), class (...",Search query:\nwattpad reading comprehension\n...
3,00dea7cb26b0df14733b1aa2e48d4189,non_rel,wattpad reading comprehension,"[304512725, 478191, 299291709, 293036332]","language (0.146), speed (0.097), reading compr...",Search query:\nwattpad reading comprehension\n...
4,69a3ac0c9cf2fb3bc0b2ceb56db2edfe,rel,synthetic biology nasa,"[78479327, 24774160, 292166320]","team (0.229), igem (0.178), brown (0.153), htt...",Search query:\nsynthetic biology nasa\n\nImpor...


doc_variant
contrastive    376
non_rel        376
rel            376
dtype: int64
